# Example single patient mrvi run for differential expression
only 1 patient is shown but all outputs provided

In [ ]:
### conda activate [] environment with scvi-tools

import scanpy as sc
from scvi.external import MRVI
import os
import pandas as pd

import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

import sys
sys.path.append('../../1_figure_CL_proof_of_concept/code/')
import utils_00 as gf_utils
large_data_dir = gf_utils.large_data_dir


In [ ]:
def add_genotype_column(adata, variant, min_counts):
    if variant + '_mutated' in adata.obsm['genotypes'].columns:
        genotype_cols = [variant + '_wt', variant + '_mutated', variant + '_heterozygous']
        genotype_data = adata.obsm['genotypes'][genotype_cols]
        
        # Handle all-NA rows to avoid FutureWarning
        all_na_mask = genotype_data.isna().all(axis=1)
        genotypes = pd.Series(index=genotype_data.index, dtype='object')
        
        # Only call idxmax on rows that have at least one non-NA value
        if not all_na_mask.all():
            genotypes[~all_na_mask] = genotype_data[~all_na_mask].idxmax(axis=1)
        
        # Clean up the column names
        genotypes = genotypes.str.replace(variant + '_', '', regex=False)
        adata.obs['genotype'] = genotypes
        
    elif variant + '_wt' in adata.obsm['genotypes'].columns:
        genotype_data = adata.obsm['genotypes'][[variant + '_wt']]
        
        # Handle all-NA rows
        all_na_mask = genotype_data.isna().all(axis=1)
        genotypes = pd.Series(index=genotype_data.index, dtype='object')
        
        if not all_na_mask.all():
            genotypes[~all_na_mask] = genotype_data[~all_na_mask].idxmax(axis=1)
        
        genotypes = genotypes.str.replace(variant + '_', '', regex=False)
        adata.obs['genotype'] = genotypes
    
    # Apply the min_counts filter
    adata.obs.loc[adata.obsm['genotypes'][variant + '_high_confidence_counts'] < min_counts, 'genotype'] = None
    return adata

def run_mrvi(adata, n_top_genes, save=False, save_dir=''):
    os.system('mkdir -p ' + save_dir)

    adata.X = adata.layers['raw_data']

    sc.pp.highly_variable_genes(
        adata, n_top_genes=n_top_genes, inplace=True, subset=True, flavor="seurat_v3"
    )

    MRVI.setup_anndata(adata, sample_key='sample_genotype')
    model = MRVI(adata)
    model.train(max_epochs=400, early_stopping=True)

    u = model.get_latent_representation()
    adata.obsm["u"] = u
    sc.pp.neighbors(adata, use_rep="u", key_added="neighbors_u")
    sc.tl.umap(adata, min_dist=0.3, neighbors_key="neighbors_u", key_added="X_umap_u")

    v = model.get_latent_representation(give_z=True)
    adata.obsm["v"] = v
    sc.pp.neighbors(adata, use_rep="v", key_added="neighbors_v")
    sc.tl.umap(adata, min_dist=0.3, neighbors_key="neighbors_v", key_added="X_umap_v")

    if save:
        adata.write(save_dir + 'mrvi.h5ad')
        model.save(save_dir + 'mrvi_model/', overwrite=True)


In [ ]:
adata_dir = large_data_dir + 'MPN_WTA/MPN_1_BC002_genotyped.h5ad'

adata = sc.read_h5ad(adata_dir)
variant = 'JAK2 c.1849G>T'

adata.obs['sample'] = 'BC002_1'
adata = add_genotype_column(adata, variant, 1)
adata = adata[(adata.obs['genotype'].notna()) & (adata.obs['genotype'] != 'heterozygous')].copy()

adata.obs['sample_genotype'] = adata.obs['sample'].astype(str) + '_' + adata.obs['genotype'].astype(str)

run_mrvi(adata, n_top_genes=2000, save=False)
